In [ ]:
import json
import re
import numpy as np
import torch
from torch_geometric.data import HeteroData
 
import pandas as pd


1. Load Data

In [ ]:
DATA_PATH = 'YOUR_DATASET_FILE.json'
 
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
 
print(f"Load complete, total records: {len(raw_data)}")

2. Build node lists and index mappings

In [ ]:
all_channels = sorted(set(
    item['summary']['頻道ID']
    for item in raw_data
    if item.get('summary', {}).get('頻道ID')
))
all_videos = sorted(set(
    item['summary']['video_id']
    for item in raw_data
    if item.get('summary', {}).get('video_id')
))
all_brands = sorted(set(
    b.strip()
    for item in raw_data
    for b in (item.get('summary', {}).get('brand名稱_標準化') or '').split(',')
    if b.strip()
))

c_map = {cid:  i for i, cid  in enumerate(all_channels)}
v_map = {vid:  i for i, vid  in enumerate(all_videos)}
b_map = {name: i for i, name in enumerate(all_brands)}

print(f'   Youtubers : {len(c_map)}')
print(f'   Videos    : {len(v_map)}')
print(f'   Brands    : {len(b_map)}')


3. Helper functions

In [ ]:
def parse_duration(duration_str: str) -> float:
    """ISO 8601 duration (e.g. 'PT1M30S') -> seconds (float)"""
    if not duration_str:
        return 0.0
    m = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_str)
    if not m:
        return 0.0
    h = int(m.group(1) or 0)
    mn = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    return float(h * 3600 + mn * 60 + s)


# Collaboration keyword list (used to classify the collaborate_with edge)
COLLAB_KEYWORDS = [
    '優惠碼', '折扣碼', '專屬碼', '優惠連結', '業配', '贊助', '廣告合作',
    '合作推廣', '合作', '聯名', '團購', '專訪', '獨家優惠',
    'sponsored', 'promo code', 'coupon', 'affiliate', 'discount code',
    'collaboration', 'partnership',
]

def classify_brand_edge(description: str) -> str:
    desc_lower = description.lower()
    for kw in COLLAB_KEYWORDS:
        if kw.lower() in desc_lower:
            return 'collaborate_with'
    return 'uses'


def get_brands(item: dict) -> list:
    raw = (item.get('summary', {}).get('brand名稱_標準化') or '').strip()
    return [b.strip() for b in raw.split(',') if b.strip()]


4. Prebuild lookup dictionaries (avoid O(n²) search)

In [ ]:
# video_id -> item
video_lookup = {
    item['summary']['video_id']: item
    for item in raw_data
    if item.get('summary', {}).get('video_id')
}
 
# channel_id -> channel_features
channel_feat_lookup: dict = {}
for item in raw_data:
    cid = item.get('summary', {}).get('頻道ID')
    if cid and cid not in channel_feat_lookup:
        channel_feat_lookup[cid] = item.get('channel_features', {})
 
# brand_name -> brand_features
brand_feat_lookup: dict = {}
for item in raw_data:
    for brand in get_brands(item):
        if brand not in brand_feat_lookup:
            brand_feat_lookup[brand] = item.get('brand_features', {})

5. Build node features

In [ ]:
# 5a. Youtuber node: Multi-level Topic Encoding (combines the original fine-grained topic with the coarse topic group)

from collections import defaultdict

channel_videos = defaultdict(list)

for item in raw_data:
    cid = item.get('summary', {}).get('頻道ID')
    vid = item.get('summary', {}).get('video_id')

    if cid and vid:
        channel_videos[cid].append(vid)


# 1. Original fine-grained topic list: for Youtuber
CHANNEL_TOPIC_LIST = [
    "Lifestyle (sociology)",
    "Entertainment",
    "Food",
    "Society",
    "Music",
    "Knowledge",
    "Film",
    "Politics",
    "Music of Asia",
    "Tourism",
    "Video game culture",
    "Television program",
    "Pop music",
    "Hobby",
    "Health",
    "Technology",
    "Role-playing video game",
    "Action game",
    "Vehicle",
    "Religion",
    "Fashion",
    "Action-adventure game",
    "Sport",
    "Business",
    "Humour",
    "Baseball",
    "Pet",
    "Physical fitness",
    "Hip hop music",
    "Military",
    "Strategy video game",
    "Association football",
    "Basketball",
    "Electronic music",
    "Others",
]

CHANNEL_TOPIC_MAP  = {t: i for i, t in enumerate(CHANNEL_TOPIC_LIST)}
NUM_CHANNEL_TOPICS = len(CHANNEL_TOPIC_LIST)
CH_OTHERS_IDX      = CHANNEL_TOPIC_MAP["Others"]


# 2. Original fine-grained topic list: for Video
TOPIC_LIST = [
    'Lifestyle (sociology)',
    'Food',
    'Politics',
    'Entertainment',
    'Society',
    'Television program',
    'Video game culture',
    'Tourism',
    'Hobby',
    'Music',
    'Film',
    'Health',
    'Vehicle',
    'Music of Asia',
    'Technology',
    'Fashion',
    'Role-playing video game',
    'Pop music',
    'Business',
    'Action-adventure game',
    'Humour',
    'Action game',
    'Pet',
    'Religion',
    'Knowledge',
    'Physical fitness',
    'Sport',
    'Baseball',
    'Simulation video game',
    'Military',
    'Strategy video game',
    'Hip hop music',
    'Performing arts',
    'Racing video game',
    'Motorsport',
    'Others',
]

TOPIC_MAP  = {topic: i for i, topic in enumerate(TOPIC_LIST)}
NUM_TOPICS = len(TOPIC_LIST)
OTHERS_IDX = TOPIC_MAP["Others"]


# 3. Coarse topic group list
TOPIC_GROUP_LIST = [
    "Lifestyle",
    "Food",
    "Travel",
    "Entertainment",
    "Music",
    "Society",
    "Gaming",
    "Technology",
    "Business",
    "Knowledge",
    "Sport",
    "Vehicle",
    "Others",
]

TOPIC_GROUP_MAP = {t: i for i, t in enumerate(TOPIC_GROUP_LIST)}
NUM_TOPIC_GROUPS = len(TOPIC_GROUP_LIST)
GROUP_OTHERS_IDX = TOPIC_GROUP_MAP["Others"]


# 4. coarse topic group mapping table

TOPIC_TO_GROUP = {
    # Lifestyle category
    "Lifestyle (sociology)": "Lifestyle",
    "Fashion": "Lifestyle",
    "Health": "Lifestyle",
    "Physical fitness": "Lifestyle",
    "Pet": "Lifestyle",
    "Hobby": "Lifestyle",

    # Food / Travel
    "Food": "Food",
    "Tourism": "Travel",

    # Entertainment category
    "Entertainment": "Entertainment",
    "Television program": "Entertainment",
    "Film": "Entertainment",
    "Humour": "Entertainment",
    "Performing arts": "Entertainment",

    # Music category
    "Music": "Music",
    "Music of Asia": "Music",
    "Pop music": "Music",
    "Hip hop music": "Music",
    "Electronic music": "Music",

    # Society category
    "Politics": "Society",
    "Society": "Society",
    "Religion": "Society",
    "Military": "Society",

    # Gaming category
    "Video game culture": "Gaming",
    "Role-playing video game": "Gaming",
    "Action game": "Gaming",
    "Action-adventure game": "Gaming",
    "Simulation video game": "Gaming",
    "Strategy video game": "Gaming",
    "Racing video game": "Gaming",

    # Technology / Business / Knowledge
    "Technology": "Technology",
    "Business": "Business",
    "Knowledge": "Knowledge",

    # Sport category
    "Sport": "Sport",
    "Baseball": "Sport",
    "Basketball": "Sport",
    "Association football": "Sport",
    "Motorsport": "Sport",

    # Vehicle category
    "Vehicle": "Vehicle",

    # fallback
    "Others": "Others",
}


def normalize_topic(raw: str) -> str:
    """
    Convert a raw YouTube official_topics entry into a clean topic name.
    Example:
    https://en.wikipedia.org/wiki/Lifestyle_(sociology)
    → Lifestyle (sociology)
    """
    if raw is None:
        return "Others"

    raw = str(raw)

    if "wikipedia.org" in raw:
        raw = raw.split("/wiki/")[-1]

    return raw.replace("_", " ")


def topic_to_group(raw_topic: str) -> str:
    """
    Convert a raw topic into its coarse topic group.
    Falls back to Others if not found in TOPIC_TO_GROUP.
    """
    topic = normalize_topic(raw_topic)
    return TOPIC_TO_GROUP.get(topic, "Others")

# 5. Youtuber node features (Final dimension:55)

youtuber_feats = []

for cid in all_channels:
    cf = channel_feat_lookup.get(cid, {})

    base = [
        np.log1p(cf.get("channel_subscribers") or 0),
        np.log1p(cf.get("channel_total_views") or 0),
        float(cf.get("uploads_per_week") or 0),
        float(cf.get("mean_like_rate") or 0),
        float(cf.get("mean_comment_rate") or 0),
        float(cf.get("shorts_ratio") or 0),
        np.log1p(cf.get("videoCount") or 0),
    ]

    # 5-1. Original fine-grained topic BoW
    ch_topic_bow = [0.0] * NUM_CHANNEL_TOPICS

    # 5-2. Coarse topic group BoW
    ch_topic_group_bow = [0.0] * NUM_TOPIC_GROUPS

    for vid in channel_videos[cid]:
        item = video_lookup.get(vid, {})
        vf = item.get("video_features", {}) if item else {}
        raw_topics = vf.get("official_topics") or []

        if not raw_topics:
            # Original topic fallback
            ch_topic_bow[CH_OTHERS_IDX] += 1.0

            # Coarse topic group fallback
            ch_topic_group_bow[GROUP_OTHERS_IDX] += 1.0

        else:
            has_unknown_original = False
            has_unknown_group = False

            for raw_t in raw_topics:
                t = normalize_topic(raw_t)

                # Original fine-grained topic
                if t in CHANNEL_TOPIC_MAP and t != "Others":
                    ch_topic_bow[CHANNEL_TOPIC_MAP[t]] += 1.0
                else:
                    has_unknown_original = True

                # Coarse topic group
                group = topic_to_group(t)
                if group in TOPIC_GROUP_MAP and group != "Others":
                    ch_topic_group_bow[TOPIC_GROUP_MAP[group]] += 1.0
                else:
                    has_unknown_group = True

            if has_unknown_original:
                ch_topic_bow[CH_OTHERS_IDX] += 1.0

            if has_unknown_group:
                ch_topic_group_bow[GROUP_OTHERS_IDX] += 1.0

    # Original fine-grained topic L1 normalize
    total_original = sum(ch_topic_bow)

    if total_original > 0:
        ch_topic_bow = [v / total_original for v in ch_topic_bow]
    else:
        ch_topic_bow[CH_OTHERS_IDX] = 1.0

    # Coarse topic group L1 normalize
    total_group = sum(ch_topic_group_bow)

    if total_group > 0:
        ch_topic_group_bow = [v / total_group for v in ch_topic_group_bow]
    else:
        ch_topic_group_bow[GROUP_OTHERS_IDX] = 1.0

    youtuber_feats.append(base + ch_topic_bow + ch_topic_group_bow)


# 5b. Video node: V4 Multi-level Topic Encoding (Final dimension: 53)

video_feats = []

for vid in all_videos:
    item = video_lookup.get(vid, {})
    vf = item.get("video_features", {}) if item else {}

    views = np.log1p(vf.get("views") or 0)
    likes = np.log1p(vf.get("likes") or 0)
    comments = np.log1p(vf.get("comments") or 0)
    duration = np.log1p(parse_duration(vf.get("duration")))

    # Original fine-grained topic multi-hot
    topic_multihot = [0.0] * NUM_TOPICS

    # Coarse topic group multi-hot
    topic_group_multihot = [0.0] * NUM_TOPIC_GROUPS

    topics = vf.get("official_topics") or []

    if not topics:
        topic_multihot[OTHERS_IDX] = 1.0
        topic_group_multihot[GROUP_OTHERS_IDX] = 1.0

    else:
        has_unknown_original = False
        has_unknown_group = False

        for raw_t in topics:
            t = normalize_topic(raw_t)

            # Original fine-grained topic multi-hot
            if t in TOPIC_MAP and t != "Others":
                topic_multihot[TOPIC_MAP[t]] = 1.0
            else:
                has_unknown_original = True

            # Coarse topic group multi-hot
            group = topic_to_group(t)
            if group in TOPIC_GROUP_MAP and group != "Others":
                topic_group_multihot[TOPIC_GROUP_MAP[group]] = 1.0
            else:
                has_unknown_group = True

        if has_unknown_original:
            topic_multihot[OTHERS_IDX] = 1.0

        if has_unknown_group:
            topic_group_multihot[GROUP_OTHERS_IDX] = 1.0

    video_feats.append([views, likes, comments, duration] + topic_multihot + topic_group_multihot)

In [ ]:
# 5c. Brand node: Industry one-hot + behavior distribution features 

from collections import defaultdict

INDUSTRY_LIST = [
    '生活居家', '食品飲料', '科技3C', '美妝時尚',
    '娛樂', '健康保健', '教育', '運動', '遊戲', '寵物', '其他'
]
INDUSTRY_MAP = {v: i for i, v in enumerate(INDUSTRY_LIST)}
NUM_INDUSTRIES = len(INDUSTRY_LIST)  # 11


def get_youtube_category(item: dict):
    """
    Try to read the YouTube category from raw_data.
    Returns None if the data has no category field; v11 will
    automatically skip this component in that case.
    """
    possible_keys = [
        'categoryId', 'category_id', 'category', 'category_name',
        'video_category_id', 'video_category',
        'youtube_category_id', 'youtube_category', 'youtube_category_name'
    ]

    for store_name in ['video_features', 'summary']:
        store = item.get(store_name, {}) or {}
        for key in possible_keys:
            val = store.get(key)
            if val is not None and str(val).strip() != '':
                return str(val).strip()
    return None


# YouTube category list: only becomes a dimension if the raw data has a category field 
youtube_categories = sorted(set(
    cat for item in raw_data
    for cat in [get_youtube_category(item)]
    if cat is not None
))
YOUTUBE_CATEGORY_MAP = {cat: i for i, cat in enumerate(youtube_categories)}
NUM_YOUTUBE_CATEGORIES = len(youtube_categories)

if NUM_YOUTUBE_CATEGORIES == 0:
    print('No YouTube category field found, v11 will not add the YouTube category distribution.')
else:
    print(f'Found {NUM_YOUTUBE_CATEGORIES} YouTube category classes')


channel_raw_topic_lookup = {}
channel_group_topic_lookup = {}

for cid in all_channels:
    ci = c_map[cid]
    y_feat = youtuber_feats[ci]

    raw_start = 7
    raw_end = raw_start + NUM_CHANNEL_TOPICS
    group_start = raw_end
    group_end = group_start + NUM_TOPIC_GROUPS

    channel_raw_topic_lookup[cid] = y_feat[raw_start:raw_end]
    channel_group_topic_lookup[cid] = y_feat[group_start:group_end]


# Initialize brand aggregation containers 
brand_video_youtube_cat_counts = defaultdict(lambda: [0.0] * NUM_YOUTUBE_CATEGORIES)
brand_video_raw_topic_counts   = defaultdict(lambda: [0.0] * NUM_TOPICS)
brand_video_group_counts       = defaultdict(lambda: [0.0] * NUM_TOPIC_GROUPS)
brand_youtuber_raw_sum         = defaultdict(lambda: [0.0] * NUM_CHANNEL_TOPICS)
brand_youtuber_group_sum       = defaultdict(lambda: [0.0] * NUM_TOPIC_GROUPS)

brand_video_count = defaultdict(float)
brand_youtuber_count = defaultdict(float)


# Aggregate content distribution for "videos the brand appears in" and "YouTubers linked to the brand" from raw_data 
for item in raw_data:
    summary = item.get('summary', {}) or {}
    cid = summary.get('頻道ID')
    brands = get_brands(item)

    if not brands:
        continue

    vf = item.get('video_features', {}) or {}
    raw_topics = vf.get('official_topics') or []

    # Video raw topic multi-hot
    video_topic_vec = [0.0] * NUM_TOPICS

    # Video topic group multi-hot
    video_group_vec = [0.0] * NUM_TOPIC_GROUPS

    if not raw_topics:
        video_topic_vec[OTHERS_IDX] = 1.0
        video_group_vec[GROUP_OTHERS_IDX] = 1.0
    else:
        has_unknown_original = False
        has_unknown_group = False

        for raw_t in raw_topics:
            t = normalize_topic(raw_t)

            if t in TOPIC_MAP and t != 'Others':
                video_topic_vec[TOPIC_MAP[t]] = 1.0
            else:
                has_unknown_original = True

            group = topic_to_group(t)
            if group in TOPIC_GROUP_MAP and group != 'Others':
                video_group_vec[TOPIC_GROUP_MAP[group]] = 1.0
            else:
                has_unknown_group = True

        if has_unknown_original:
            video_topic_vec[OTHERS_IDX] = 1.0
        if has_unknown_group:
            video_group_vec[GROUP_OTHERS_IDX] = 1.0

    # YouTube category one-hot 
    youtube_cat = get_youtube_category(item)

    # Youtuber topic distribution
    y_raw_vec = channel_raw_topic_lookup.get(cid)
    y_group_vec = channel_group_topic_lookup.get(cid)

    for brand in brands:
        if brand not in b_map:
            continue

        # Topic distribution of videos the brand appears in
        for j, val in enumerate(video_topic_vec):
            brand_video_raw_topic_counts[brand][j] += val
        for j, val in enumerate(video_group_vec):
            brand_video_group_counts[brand][j] += val
        brand_video_count[brand] += 1.0

        # YouTube category distribution of videos the brand appears in
        if NUM_YOUTUBE_CATEGORIES > 0 and youtube_cat in YOUTUBE_CATEGORY_MAP:
            brand_video_youtube_cat_counts[brand][YOUTUBE_CATEGORY_MAP[youtube_cat]] += 1.0

        # Category / topic distribution of YouTubers linked to the brand
        if y_raw_vec is not None and y_group_vec is not None:
            for j, val in enumerate(y_raw_vec):
                brand_youtuber_raw_sum[brand][j] += val
            for j, val in enumerate(y_group_vec):
                brand_youtuber_group_sum[brand][j] += val
            brand_youtuber_count[brand] += 1.0


def normalize_count_vector(vec, fallback_idx=None):
    """Convert a count vector into an L1 distribution; if all zero, use fallback_idx if given."""
    total = float(sum(vec))
    if total > 0:
        return [float(v) / total for v in vec]
    out = [0.0] * len(vec)
    if fallback_idx is not None and len(out) > 0:
        out[fallback_idx] = 1.0
    return out


brand_feats = []
brand_industry_labels = []

# Component slice registry, useful for later validation and thesis documentation
BRAND_FEATURE_SLICES = {}
start = 0

def register_component(name, dim):
    global start
    BRAND_FEATURE_SLICES[name] = (start, start + dim)
    start += dim

register_component('industry_onehot', NUM_INDUSTRIES)
register_component('video_youtube_category_dist', NUM_YOUTUBE_CATEGORIES)
register_component('video_raw_topic_dist', NUM_TOPICS)
register_component('video_topic_group_dist', NUM_TOPIC_GROUPS)
register_component('youtuber_raw_topic_dist', NUM_CHANNEL_TOPICS)
register_component('youtuber_topic_group_dist', NUM_TOPIC_GROUPS)

BRAND_FEATURE_DIM = start

for brand in all_brands:
    bf = brand_feat_lookup.get(brand, {})

    # 1. Industry one-hot
    label = bf.get('industry_final') or '其他'  # '其他' = Others, matches raw data label
    idx = INDUSTRY_MAP.get(label, INDUSTRY_MAP['其他'])

    industry_onehot = [0.0] * NUM_INDUSTRIES
    industry_onehot[idx] = 1.0

    # 2. YouTube category distribution of videos the brand appears in
    if NUM_YOUTUBE_CATEGORIES > 0:
        video_youtube_cat_dist = normalize_count_vector(
            brand_video_youtube_cat_counts[brand],
            fallback_idx=None
        )
    else:
        video_youtube_cat_dist = []

    # 3. Raw topic distribution of videos the brand appears in
    video_raw_topic_dist = normalize_count_vector(
        brand_video_raw_topic_counts[brand],
        fallback_idx=OTHERS_IDX
    )

    # 4. Topic group distribution of videos the brand appears in
    video_topic_group_dist = normalize_count_vector(
        brand_video_group_counts[brand],
        fallback_idx=GROUP_OTHERS_IDX
    )

    # 5. Raw topic distribution of YouTubers linked to the brand
    youtuber_raw_topic_dist = normalize_count_vector(
        brand_youtuber_raw_sum[brand],
        fallback_idx=CH_OTHERS_IDX
    )

    # 6. Topic group distribution of YouTubers linked to the brand
    youtuber_topic_group_dist = normalize_count_vector(
        brand_youtuber_group_sum[brand],
        fallback_idx=GROUP_OTHERS_IDX
    )

    feat = (
        industry_onehot
        + video_youtube_cat_dist
        + video_raw_topic_dist
        + video_topic_group_dist
        + youtuber_raw_topic_dist
        + youtuber_topic_group_dist
    )

    assert len(feat) == BRAND_FEATURE_DIM, (
        f'Brand feature dimension mismatch: {len(feat)} != {BRAND_FEATURE_DIM}'
    )

    brand_feats.append(feat)
    brand_industry_labels.append(label)

print('Brand feature v11 complete')
print(f'   brand feature dim = {BRAND_FEATURE_DIM}')
print('   feature components:')
for name, (s, e) in BRAND_FEATURE_SLICES.items():
    print(f'   - {name:<30}: [{s:>3}, {e:>3}) dim={e-s}')
print('   relation type proportion: not added (to avoid leakage)')


## 5d. Brand Feature Cleaning

This step generates a full column-diagnostics table after the v11 brand features are built, then removes non-discriminative behavior features according to fixed rules.

Cleaning rules:

- `industry_onehot` is always kept.
- Behavior features that are all-zero / constant columns are removed.
- Behavior features with too low a usage rate are removed.
- Behavior features with an excessively high mean (i.e. almost every brand is close to 1 on that dimension) are removed.

In [ ]:
# 5d. v14 Brand Feature Cleaning
import pandas as pd
import torch
import torch.nn.functional as F

# ---------- Adjustable parameters ----------
INDUSTRY_COMPONENT_NAME = 'industry_onehot'
STD_EPS = 1e-8
MIN_NONZERO_RATIO = 0.001      
DOMINANT_MEAN_THRESHOLD = 0.95 

REPORT_FULL_PATH = 'v14_brand_feature_cleaning_full_report.csv'
REPORT_REMOVED_PATH = 'v14_removed_brand_features.csv'
REPORT_KEPT_PATH = 'v14_kept_brand_features.csv'

brand_tensor_v11 = torch.tensor(brand_feats, dtype=torch.float)
num_brands, original_brand_dim = brand_tensor_v11.shape

print('═' * 70)
print('v14 Brand Feature Cleaning')
print('═' * 70)
print(f'Original v11 brand feature shape: {tuple(brand_tensor_v11.shape)}')
print(f'Cleaning rule: keep industry; for behavior keep std>{STD_EPS}, nonzero_ratio>={MIN_NONZERO_RATIO}, mean<={DOMINANT_MEAN_THRESHOLD}')

# Build a component-name mapping for each global feature index
feature_component_names = [None] * original_brand_dim
feature_local_indices = [None] * original_brand_dim

for comp_name, (s, e) in BRAND_FEATURE_SLICES.items():
    for local_i, global_i in enumerate(range(s, e)):
        feature_component_names[global_i] = comp_name
        feature_local_indices[global_i] = local_i

# Column statistics
col_mean = brand_tensor_v11.mean(dim=0)
col_std = brand_tensor_v11.std(dim=0)
col_min = brand_tensor_v11.min(dim=0).values
col_max = brand_tensor_v11.max(dim=0).values
nonzero_ratio = (brand_tensor_v11 != 0).float().mean(dim=0)

keep_mask = torch.ones(original_brand_dim, dtype=torch.bool)
remove_reasons = []

for j in range(original_brand_dim):
    comp = feature_component_names[j]
    reasons = []

    if comp == INDUSTRY_COMPONENT_NAME:
        # industry one-hot is always kept
        keep = True
        reasons.append('keep_industry_onehot')
    else:
        keep = True

        if col_std[j].item() <= STD_EPS:
            keep = False
            reasons.append('constant_or_all_zero')

        if nonzero_ratio[j].item() < MIN_NONZERO_RATIO:
            keep = False
            reasons.append('too_sparse')

        if col_mean[j].item() > DOMINANT_MEAN_THRESHOLD:
            keep = False
            reasons.append('dominant_mean_gt_threshold')

        if keep:
            reasons.append('keep_behavior')

    keep_mask[j] = keep
    remove_reasons.append(';'.join(reasons))

# Output the full report 
report_df = pd.DataFrame({
    'global_feature_index': list(range(original_brand_dim)),
    'component': feature_component_names,
    'local_index_in_component': feature_local_indices,
    'mean': col_mean.numpy(),
    'std': col_std.numpy(),
    'min': col_min.numpy(),
    'max': col_max.numpy(),
    'nonzero_ratio': nonzero_ratio.numpy(),
    'keep': keep_mask.numpy(),
    'reason': remove_reasons,
})

report_df.to_csv(REPORT_FULL_PATH, index=False, encoding='utf-8-sig')
report_df[~report_df['keep']].to_csv(REPORT_REMOVED_PATH, index=False, encoding='utf-8-sig')
report_df[report_df['keep']].to_csv(REPORT_KEPT_PATH, index=False, encoding='utf-8-sig')

print('\n[Feature Cleaning Summary]')
print(f'Original dim : {original_brand_dim}')
print(f'Kept dim     : {int(keep_mask.sum().item())}')
print(f'Removed dim  : {int((~keep_mask).sum().item())}')
print(f'Reports saved:')
print(f'  - {REPORT_FULL_PATH}')
print(f'  - {REPORT_REMOVED_PATH}')
print(f'  - {REPORT_KEPT_PATH}')

print('\n[Removed feature count by reason]')
removed_df = report_df[~report_df['keep']].copy()
if len(removed_df) > 0:
    for reason, cnt in removed_df['reason'].value_counts().items():
        print(f'  {reason:<45}: {cnt}')
else:
    print('  No feature removed.')

print('\n[Kept feature count by component]')
print(report_df[report_df['keep']]['component'].value_counts().to_string())

#  Update brand_feats 
brand_tensor_v14 = brand_tensor_v11[:, keep_mask]
brand_feats = brand_tensor_v14.tolist()
BRAND_FEATURE_DIM_ORIGINAL_V11 = original_brand_dim
BRAND_FEATURE_DIM = brand_tensor_v14.shape[1]
BRAND_FEATURE_KEEP_MASK = keep_mask
BRAND_FEATURE_CLEANING_REPORT = report_df

# Rebuild BRAND_FEATURE_SLICES after cleaning
old_slices = dict(BRAND_FEATURE_SLICES)
new_slices = {}
new_start = 0
kept_global_indices_by_component = {}

for comp_name, (s, e) in old_slices.items():
    original_indices = list(range(s, e))
    kept_indices = [idx for idx in original_indices if bool(keep_mask[idx].item())]
    dim = len(kept_indices)
    new_slices[comp_name] = (new_start, new_start + dim)
    kept_global_indices_by_component[comp_name] = kept_indices
    new_start += dim

BRAND_FEATURE_SLICES_ORIGINAL_V11 = old_slices
BRAND_FEATURE_SLICES = new_slices
BRAND_FEATURE_KEPT_GLOBAL_INDICES_BY_COMPONENT = kept_global_indices_by_component

print('\n[Updated brand feature components after cleaning]')
for name, (s, e) in BRAND_FEATURE_SLICES.items():
    print(f'  {name:<30}: [{s:>3}, {e:>3}) dim={e-s}')

# Raw cosine comparison before/after cleaning
def cosine_summary_tensor(X, label):
    X = X.float()
    Xn = F.normalize(X, p=2, dim=1)
    cos = Xn @ Xn.T
    mask = ~torch.eye(cos.size(0), dtype=torch.bool, device=cos.device)
    vals = cos[mask].detach().cpu().numpy()
    print(f'\n[{label}]')
    print(f'  Mean cosine : {vals.mean():.4f}')
    print(f'  Std cosine  : {vals.std():.4f}')
    print(f'  Min cosine  : {vals.min():.4f}')
    print(f'  Max cosine  : {vals.max():.4f}')
    print(f'  % > 0.5     : {(vals > 0.5).mean()*100:.2f}%')
    print(f'  % > 0.8     : {(vals > 0.8).mean()*100:.2f}%')
    print(f'  % > 0.95    : {(vals > 0.95).mean()*100:.2f}%')
    return vals

_ = cosine_summary_tensor(brand_tensor_v11, 'Before cleaning: v11 brand feature')
_ = cosine_summary_tensor(brand_tensor_v14, 'After cleaning: v14 brand feature')


Validation

In [ ]:
import torch
import numpy as np

youtuber_tensor = torch.tensor(youtuber_feats, dtype=torch.float)
video_tensor    = torch.tensor(video_feats,    dtype=torch.float)
brand_tensor    = torch.tensor(brand_feats,    dtype=torch.float)

print("=" * 60)
print("Node feature validation: V4 Multi-level Topic Encoding + Brand feature cleaning v14")
print("=" * 60)


# 【Youtuber】
print("\n[Youtuber]")
expected_youtuber_dim = 7 + NUM_CHANNEL_TOPICS + NUM_TOPIC_GROUPS

# 1. Shape
assert youtuber_tensor.shape == (len(all_channels), expected_youtuber_dim), (
    f"Shape mismatch: got {tuple(youtuber_tensor.shape)}, "
    f"expected {(len(all_channels), expected_youtuber_dim)}"
)
print(f"Shape : {tuple(youtuber_tensor.shape)}")

# 2. No NaN / Inf
assert not torch.isnan(youtuber_tensor).any(), "Contains NaN"
assert not torch.isinf(youtuber_tensor).any(), "Contains Inf"
print("NaN/Inf : none")

# 3. The base part (first 7 dims) should be non-negative
base_part = youtuber_tensor[:, :7]

assert base_part.min() >= 0, "base has negative values"
print(f"base range: [{base_part.min():.4f}, {base_part.max():.4f}]")

# 4. Original fine-grained topic proportion
raw_topic_start = 7
raw_topic_end   = 7 + NUM_CHANNEL_TOPICS

raw_topic_part = youtuber_tensor[:, raw_topic_start:raw_topic_end]
raw_row_sums   = raw_topic_part.sum(dim=1)

assert torch.allclose(raw_row_sums, torch.ones_like(raw_row_sums), atol=1e-5), (
    "Original fine-grained topic proportion row sums are not 1"
)
print(
    f"  Original fine-grained topic row sums: "
    f"min={raw_row_sums.min():.4f} max={raw_row_sums.max():.4f}"
)

assert raw_topic_part.min() >= 0 and raw_topic_part.max() <= 1, (
    "Original fine-grained topic proportion range is abnormal"
)
print(
    f"  Original fine-grained topic range: "
    f"[{raw_topic_part.min():.4f}, {raw_topic_part.max():.4f}]"
)

# 5. Topic group proportion
group_topic_start = raw_topic_end
group_topic_end   = raw_topic_end + NUM_TOPIC_GROUPS

group_topic_part = youtuber_tensor[:, group_topic_start:group_topic_end]
group_row_sums   = group_topic_part.sum(dim=1)

assert torch.allclose(group_row_sums, torch.ones_like(group_row_sums), atol=1e-5), (
    "Topic Group proportion row sums are not 1"
)
print(
    f"  Topic Group row sums: "
    f"min={group_row_sums.min():.4f} max={group_row_sums.max():.4f}"
)

assert group_topic_part.min() >= 0 and group_topic_part.max() <= 1, (
    "Topic Group proportion range is abnormal"
)
print(
    f"  Topic Group range: "
    f"[{group_topic_part.min():.4f}, {group_topic_part.max():.4f}]"
)

# 6. Channels with no videos should fallback to raw Others=1.0, Group Others=1.0
empty_ch = [
    i for i, cid in enumerate(all_channels)
    if len(channel_videos[cid]) == 0
]

print(f"  Channels with no videos: {len(empty_ch)}", end="")

if empty_ch:
    for i in empty_ch[:3]:
        assert raw_topic_part[i][CH_OTHERS_IDX].item() == 1.0, (
            f"idx={i} original topic fallback failed, Others != 1"
        )
        assert group_topic_part[i][GROUP_OTHERS_IDX].item() == 1.0, (
            f"idx={i} Topic Group fallback failed, Others != 1"
        )

print("")

# 7. Spot-check one channel
sample_idx = 0
sample_cid = all_channels[sample_idx]

print(f"\n  [Spot-check] channel={sample_cid}, video count={len(channel_videos[sample_cid])}")

print("    Original fine-grained topic proportion:")
for topic, val in zip(CHANNEL_TOPIC_LIST, raw_topic_part[sample_idx].tolist()):
    if val > 0:
        print(f"      {topic:<30} {val:.4f}")

print("    Topic Group proportion:")
for topic_group, val in zip(TOPIC_GROUP_LIST, group_topic_part[sample_idx].tolist()):
    if val > 0:
        print(f"      {topic_group:<30} {val:.4f}")


# 【Video】
print("\n[Video]")
expected_video_dim = 4 + NUM_TOPICS + NUM_TOPIC_GROUPS

# 1. Shape
assert video_tensor.shape == (len(all_videos), expected_video_dim), (
    f"Shape mismatch: got {tuple(video_tensor.shape)}, "
    f"expected {(len(all_videos), expected_video_dim)}"
)
print(f"  Shape     : {tuple(video_tensor.shape)} ")

# 2. No NaN / Inf
assert not torch.isnan(video_tensor).any(), "Contains NaN"
assert not torch.isinf(video_tensor).any(), "Contains Inf"
print("  NaN/Inf   : none ")

# 3. Numerical features (first 4 dims) should be non-negative
num_part = video_tensor[:, :4]

assert num_part.min() >= 0, "Numerical features have negative values"
print(f"  Numerical range: [{num_part.min():.4f}, {num_part.max():.4f}] ")

# 4. Original fine-grained topic multi-hot
raw_video_start = 4
raw_video_end   = 4 + NUM_TOPICS

raw_topic_part_v = video_tensor[:, raw_video_start:raw_video_end]
unique_raw_vals  = raw_topic_part_v.unique()

assert set(unique_raw_vals.tolist()).issubset({0.0, 1.0}), (
    f"Original fine-grained topic multi-hot has non-0/1 values: {unique_raw_vals}"
)
print("  Original fine-grained topic multi-hot range: 0/1 only ")

raw_row_sums_v = raw_topic_part_v.sum(dim=1)

assert (raw_row_sums_v >= 1).all(), (
    "Some video has all-zero original fine-grained topics"
)
print(
    f"  Original fine-grained topic: every row has at least one 1: "
    f"(max={int(raw_row_sums_v.max())} topics/video)"
)

# 5. Topic group multi-hot
group_video_start = raw_video_end
group_video_end   = raw_video_end + NUM_TOPIC_GROUPS

group_topic_part_v = video_tensor[:, group_video_start:group_video_end]
unique_group_vals  = group_topic_part_v.unique()

assert set(unique_group_vals.tolist()).issubset({0.0, 1.0}), (
    f"Topic Group multi-hot has non-0/1 values: {unique_group_vals}"
)
print("  Topic Group multi-hot range: 0/1 only ")

group_row_sums_v = group_topic_part_v.sum(dim=1)

assert (group_row_sums_v >= 1).all(), (
    "Some video has all-zero Topic Group"
)
print(
    f"  Topic Group: every row has at least one 1: "
    f"(max={int(group_row_sums_v.max())} groups/video)"
)

# 6. Spot-check one video
sample_vid = all_videos[0]
sample_item = video_lookup.get(sample_vid, {})
sample_topics = sample_item.get("video_features", {}).get("official_topics") or []

print(f"\n  [Spot-check] video={sample_vid}")
print(f"    raw topics : {sample_topics}")

print("    encoded original topics : ", end="")
for topic, val in zip(TOPIC_LIST, raw_topic_part_v[0].tolist()):
    if val > 0:
        print(topic, end="  ")
print()

print("    encoded topic groups    : ", end="")
for topic_group, val in zip(TOPIC_GROUP_LIST, group_topic_part_v[0].tolist()):
    if val > 0:
        print(topic_group, end="  ")
print()


# 【Brand】
print("\n[Brand]")

# v14: industry one-hot + cleaned behavior distribution features
expected_brand_dim = BRAND_FEATURE_DIM

# 1. Shape
assert brand_tensor.shape == (len(all_brands), expected_brand_dim), (
    f"Shape mismatch: got {tuple(brand_tensor.shape)}, "
    f"expected {(len(all_brands), expected_brand_dim)}"
)
print(f"  Shape     : {tuple(brand_tensor.shape)} ")

# 2. No NaN / Inf
assert not torch.isnan(brand_tensor).any(), "Contains NaN"
assert not torch.isinf(brand_tensor).any(), "Contains Inf"
print("NaN/Inf : none")

# 3. Check the industry one-hot block
s, e = BRAND_FEATURE_SLICES['industry_onehot']
onehot_part = brand_tensor[:, s:e]
row_sums_b  = onehot_part.sum(dim=1)

assert torch.allclose(row_sums_b, torch.ones_like(row_sums_b), atol=1e-5), (
    "industry one-hot row sums are not 1"
)
unique_vals_b = onehot_part.unique()
assert set(unique_vals_b.tolist()).issubset({0.0, 1.0}), (
    f"industry one-hot has non-0/1 values: {unique_vals_b}"
)
print(f"  industry one-hot: dim={e-s}, row sum min={row_sums_b.min():.1f}, max={row_sums_b.max():.1f} ")

# 4. Check the value range and row sums for every distribution block
for comp_name, (s, e) in BRAND_FEATURE_SLICES.items():
    part = brand_tensor[:, s:e]
    dim = e - s
    if dim == 0:
        print(f"  {comp_name:<30}: dim=0, field not present in the data, skipping")
        continue

    assert part.min() >= -1e-6, f"{comp_name} has negative values"
    assert part.max() <= 1.0 + 1e-6, f"{comp_name} has values greater than 1"

    if comp_name != 'industry_onehot':
        row_sum = part.sum(dim=1)
        print(
            f"  {comp_name:<30}: dim={dim:<3} "
            f"row_sum mean={row_sum.mean():.4f}, min={row_sum.min():.4f}, max={row_sum.max():.4f} "
        )

# 5. Distribution per industry
print("\n  [Industry distribution]")
industry_counts = onehot_part.sum(dim=0).int().tolist()
for name, cnt in zip(INDUSTRY_LIST, industry_counts):
    print(f"    {name:<8} : {cnt}")

# 6. Spot-check one brand feature component
sample_brand = all_brands[0]
sample_idx = b_map[sample_brand]
print(f"\n  [Spot-check] brand={sample_brand}")
for comp_name, (s, e) in BRAND_FEATURE_SLICES.items():
    if e - s == 0:
        continue
    part = brand_tensor[sample_idx, s:e]
    nonzero = (part > 0).sum().item()
    print(f"    {comp_name:<30}: dim={e-s:<3}, nonzero={nonzero}")

print("\n" + "=" * 60)
print("All validations passed")
print("=" * 60)


6. Build edges (deduplicated via set)

In [ ]:
edges_uploads       = set()   # (youtuber_idx, video_idx)
edges_uploaded_by   = set()   # (video_idx, youtuber_idx)
edges_mentions      = set()   # (video_idx, brand_idx)
edges_uses          = set()   # (youtuber_idx, brand_idx)
edges_collab        = set()   # (youtuber_idx, brand_idx)
edges_collab_rev    = set()   # (brand_idx, youtuber_idx)
 
collab_count = 0
uses_count   = 0
no_desc_count = 0
 
for item in raw_data:
    summary = item.get('summary', {})
    vid = summary.get('video_id')
    cid = summary.get('頻道ID')
    if not vid or not cid:
        continue
 
    vi = v_map[vid]
    ci = c_map[cid]
    description = item.get('video_features', {}).get('description')
    brands = get_brands(item)
 
    # uploads 
    edges_uploads.add((ci, vi))
    edges_uploaded_by.add((vi, ci))
 
    for brand in brands:
        if brand not in b_map:
            continue
        bi = b_map[brand]
 
        # mentions: video → brand
        edges_mentions.add((vi, bi))
 
        # uses / collaborate_with
        if description is None:
            no_desc_count += 1
            continue  # No description, cannot classify, skip
 
        edge_type = classify_brand_edge(description)
        if edge_type == 'collaborate_with':
            edges_collab.add((ci, bi))
            edges_collab_rev.add((bi, ci))
            collab_count += 1
        else:
            edges_uses.add((ci, bi))
            uses_count += 1
 
print(f"\n Edge statistics (after deduplication):")
print(f"   uploads          : {len(edges_uploads)}")
print(f"   uploaded_by      : {len(edges_uploaded_by)}")
print(f"   mentions         : {len(edges_mentions)}")
print(f"   uses             : {len(edges_uses)}")
print(f"   collaborate_with : {len(edges_collab)}")
print(f"   partner_with  : {len(edges_collab_rev)}")
print(f"   (skipped brand-video pairs without a description: {no_desc_count})")
 

In [ ]:
SIMILAR_BRAND_K = 1
SIMILAR_BRAND_MIN_SIM = 0.8
SIMILAR_BRAND_ALLOW_FALLBACK = True


def add_similar_brand_edges(hetero_data, K=SIMILAR_BRAND_K,
                            min_sim=SIMILAR_BRAND_MIN_SIM,
                            allow_fallback=SIMILAR_BRAND_ALLOW_FALLBACK):
    import networkx as nx
    import torch
    import torch.nn.functional as F

    G = nx.Graph()
    for ntype, store in hetero_data.x_dict.items():
        for i in range(store.shape[0]):
            G.add_node((ntype, i))
    for etype, ei in hetero_data.edge_index_dict.items():
        src_type, _, dst_type = etype
        for s, d in zip(ei[0].tolist(), ei[1].tolist()):
            G.add_edge((src_type, s), (dst_type, d))

    main_comp = max(nx.connected_components(G), key=len)
    main_brands = {idx for (ntype, idx) in main_comp if ntype == 'brand'}
    all_brand_idx = set(range(hetero_data['brand'].x.shape[0]))
    isolated = all_brand_idx - main_brands

    if not isolated:
        print("[similar_brand v11] No isolated brands, skipping")
        return hetero_data

    feat = hetero_data['brand'].x.float()
    feat_norm = F.normalize(feat, dim=1)
    src_list, dst_list = [], []
    fallback_count = 0

    candidate_mask = torch.ones(feat.shape[0], dtype=torch.bool)
    candidate_mask[list(isolated)] = False

    for iso in sorted(isolated):
        sims = feat_norm[iso] @ feat_norm.T
        sims[~candidate_mask] = -1.0
        sims[iso] = -1.0

        # Prioritize candidate brands in the same industry or reaching min_sim
        valid_idx = torch.where(sims >= min_sim)[0]

        if valid_idx.numel() > 0:
            valid_scores = sims[valid_idx]
            k_use = min(K, valid_idx.numel())
            nb_indices = valid_idx[valid_scores.topk(k_use).indices].tolist()
        elif allow_fallback:
            # If there is no non-isolated brand in the same industry, still connect
            # to top-1 so the isolated brand can enter the main graph
            nb_indices = [int(sims.argmax().item())]
            fallback_count += 1
        else:
            nb_indices = []

        for nb in nb_indices:
            src_list += [iso, nb]
            dst_list += [nb, iso]

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    hetero_data[('brand', 'similar_brand', 'brand')].edge_index = edge_index

    print(f"[similar_brand v11] Number of isolated brands: {len(isolated)}")
    print(f"[similar_brand v11] K={K}, min_sim={min_sim}, fallback={allow_fallback}")
    print(f"[similar_brand v11] Edges added: {edge_index.shape[1]} (counted bidirectionally)")
    print(f"[similar_brand v11] fallback count: {fallback_count}")

    return hetero_data



7. Assemble HeteroData

In [ ]:
def set_to_edge_index(edge_set: set) -> torch.Tensor:
    """set of (src, dst) tuples → edge_index tensor [2, E]"""
    if not edge_set:
        return torch.zeros((2, 0), dtype=torch.long)
    src, dst = zip(*edge_set)
    return torch.tensor([list(src), list(dst)], dtype=torch.long)
 
 
data = HeteroData()
 
# Node features
data['youtuber'].x = torch.tensor(youtuber_feats, dtype=torch.float)
data['video'].x    = torch.tensor(video_feats,    dtype=torch.float)
data['brand'].x    = torch.tensor(brand_feats,    dtype=torch.float)
 
# Edges
data['youtuber', 'uploads',          'video'   ].edge_index = set_to_edge_index(edges_uploads)
data['video',    'uploaded_by',      'youtuber'].edge_index = set_to_edge_index(edges_uploaded_by)
data['video',    'mentions',         'brand'   ].edge_index = set_to_edge_index(edges_mentions)
data['youtuber', 'uses',             'brand'   ].edge_index = set_to_edge_index(edges_uses)
data['youtuber', 'collaborate_with', 'brand'   ].edge_index = set_to_edge_index(edges_collab)
data['brand',    'partner_with',  'youtuber'].edge_index = set_to_edge_index(edges_collab_rev)

data = add_similar_brand_edges(data, K=SIMILAR_BRAND_K,
                               min_sim=SIMILAR_BRAND_MIN_SIM,
                               allow_fallback=SIMILAR_BRAND_ALLOW_FALLBACK)

## Check: Cleaned Brand Behavior Features and Low-Density similar_brand

This cell verifies that v14 feature cleaning took effect:

1. Whether `brand.x` contains the industry one-hot and behavior distribution features.
2. Whether `relation type proportion` was NOT added to the brand features.
3. Whether `similar_brand` remains low-density.


In [ ]:
# sanity check
print('═' * 60)
print('v14 Sanity Check')
print('═' * 60)

print(f"brand.x shape: {tuple(data['brand'].x.shape)}")
print(f"Expected cleaned brand dim = BRAND_FEATURE_DIM = {BRAND_FEATURE_DIM}")
assert data['brand'].x.shape[1] == BRAND_FEATURE_DIM, 'brand.x dim does not equal BRAND_FEATURE_DIM'

print('\n[Brand feature components]')
for name, (s, e) in BRAND_FEATURE_SLICES.items():
    print(f"  {name:<30}: [{s:>3}, {e:>3}) dim={e-s}")

# Check the industry one-hot block
s, e = BRAND_FEATURE_SLICES['industry_onehot']
industry_part = data['brand'].x[:, s:e]
brand_row_sums = industry_part.sum(dim=1)
assert torch.allclose(brand_row_sums, torch.ones_like(brand_row_sums), atol=1e-5), 'brand industry one-hot row sum is not 1'
print('\n brand industry one-hot block is normal')
print('relation type proportion was not added to brand features')
print('behavior features have been cleaned per the v14 rules')

# Check feature cosine similarity, confirming we didn't regress to v9's high-similarity issue
brand_x = data['brand'].x.float()
brand_x_norm = torch.nn.functional.normalize(brand_x, p=2, dim=1)
cos_mat = brand_x_norm @ brand_x_norm.T
n_brand = cos_mat.size(0)
mask = ~torch.eye(n_brand, dtype=torch.bool, device=cos_mat.device)
cos_values = cos_mat[mask].detach().cpu().numpy()

print('\n[Raw Brand Feature Cosine Similarity]')
print(f"  mean cosine : {cos_values.mean():.4f}")
print(f"  std cosine  : {cos_values.std():.4f}")
print(f"  % > 0.95    : {(cos_values > 0.95).mean() * 100:.2f}%")

sim_etype = ('brand', 'similar_brand', 'brand')
if sim_etype in data.edge_types:
    sim_edges = data[sim_etype].edge_index
    num_edges = sim_edges.shape[1]
    print(f"\nsimilar_brand edges: {num_edges}")

    src, dst = sim_edges
    total_deg = torch.bincount(torch.cat([src, dst]), minlength=data['brand'].num_nodes)
    print(f"similar_brand average total degree: {total_deg.float().mean().item():.4f}")
    print(f"similar_brand max total degree    : {int(total_deg.max().item())}")
else:
    print('similar_brand edge does not exist')


8. Verify output

In [ ]:
print('\n' + '═'*50)
print('HeteroData v14 construction complete')
print('═'*50)
print(data)
print()
print(f'Node feature dimensions:')
youtuber_topic_dim = data["youtuber"].x.shape[1] - 7
video_topic_dim = data["video"].x.shape[1] - 4
print(f'  youtuber.x : {data["youtuber"].x.shape}   (7 numerical + {youtuber_topic_dim} dim topic encoding)')
print(f'  video.x    : {data["video"].x.shape}  (4 numerical + {video_topic_dim} dim topic encoding)')
print(f'  brand.x    : {data["brand"].x.shape}    (v14: industry one-hot + cleaned behavior distribution features)')
print()
print('brand feature components:')
for name, (s, e) in BRAND_FEATURE_SLICES.items():
    print(f'  {name:<30}: dim={e-s}')
print()
print(f'Edge counts:')
for edge_type in data.edge_types:
    e = data[edge_type].edge_index
    print(f'  {edge_type}: {e.shape[1]}')


---
## v15: Youtuber & Video Feature Cleaning + StandardScaler Normalization

Building on the v14 HeteroData, apply the following to **youtuber** and **video** nodes:

1. **Remove zero-information features** (columns with 0% nonzero across the whole dataset)
2. **Remove extreme outliers** (youtuber F002: uploads_per_week has a max value of 714)
3. **Remove duplicate features** (youtuber F054 = F041; video F052 ≈ F039)
4. **StandardScaler normalization** (applied to all three node types)

Brand features were already handled by v14 cleaning; this version only applies normalization to them.
Output: `hetero_data15.pt`


In [ ]:
# Step 1: detect and remove zero-information features
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler

def get_drop_indices(x_tensor, extra_drop=None):
    """
    Automatically detect columns with nonzero=0%, plus manually specified
    extra_drop indices. Returns the set of indices to remove.
    """
    X = x_tensor.numpy()
    zero_cols = set(int(i) for i in np.where((X != 0).sum(axis=0) == 0)[0])
    return zero_cols | (extra_drop or set())


# Youtuber 
# F002 = uploads_per_week (std=20.26, max=714.82, extreme outlier)
# F054 = topic_group[Others], exactly duplicates F041 = raw_topic[Others]
youtuber_x = data['youtuber'].x
youtuber_drop = get_drop_indices(youtuber_x, extra_drop={2, 54})
youtuber_keep = sorted(set(range(youtuber_x.shape[1])) - youtuber_drop)

# Video 
# F052 = topic_group[Others], nearly duplicates F039 = raw_topic[Others]
video_x = data['video'].x
video_drop = get_drop_indices(video_x, extra_drop={52})
video_keep = sorted(set(range(video_x.shape[1])) - video_drop)

# Brand 
# Already cleaned in v14; this version only does normalization
brand_keep = list(range(data['brand'].x.shape[1]))

print('═' * 60)
print('v15 Feature Selection Summary')
print('═' * 60)
print(f'[youtuber] original={youtuber_x.shape[1]}, dropped={len(youtuber_drop)}, kept={len(youtuber_keep)}')
print(f'  dropped indices: {sorted(youtuber_drop)}')
print(f'[video]    original={video_x.shape[1]}, dropped={len(video_drop)}, kept={len(video_keep)}')
print(f'  dropped indices: {sorted(video_drop)}')
print(f'[brand]    original={data["brand"].x.shape[1]}, dropped=0 (already cleaned in v14), kept={len(brand_keep)}')


In [ ]:
# Step 2: StandardScaler normalization 
def clean_and_normalize(x_tensor, keep_idx):
    """
    1. Select the keep_idx columns
    2. StandardScaler (mean=0, std=1)
    Returns (normalized float32 tensor, fitted scaler)
    """
    X = x_tensor.numpy()[:, keep_idx]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float32)
    return torch.tensor(X_scaled), scaler


# Process all three node types
data_v15 = data  # Update in place on the same object

data_v15['youtuber'].x, scaler_youtuber = clean_and_normalize(data['youtuber'].x, youtuber_keep)
data_v15['video'].x,    scaler_video    = clean_and_normalize(data['video'].x,    video_keep)
data_v15['brand'].x,    scaler_brand    = clean_and_normalize(data['brand'].x,    brand_keep)

print('Normalization complete')
print(f'  youtuber.x : {data_v15["youtuber"].x.shape}')
print(f'  video.x    : {data_v15["video"].x.shape}')
print(f'  brand.x    : {data_v15["brand"].x.shape}')
print()

# Quick check: no NaN / Inf
for ntype in ['youtuber', 'video', 'brand']:
    x = data_v15[ntype].x
    assert not torch.isnan(x).any(), f'{ntype} has NaN!'
    assert not torch.isinf(x).any(), f'{ntype} has Inf!'
    print(f'[{ntype}] mean={x.mean():.4f}, std={x.std():.4f}, min={x.min():.4f}, max={x.max():.4f}')


In [ ]:
# Step 3: brand cosine validation (confirm no degradation after normalization) 
import torch.nn.functional as F

brand_x_v15 = data_v15['brand'].x.float()
brand_norm = F.normalize(brand_x_v15, p=2, dim=1)
cos_mat = brand_norm @ brand_norm.T
n = cos_mat.size(0)
mask = ~torch.eye(n, dtype=torch.bool)
cos_vals = cos_mat[mask].detach().cpu().numpy()

print('═' * 60)
print('v15 Brand Cosine Similarity (after normalization)')
print('═' * 60)
print(f'  Mean : {cos_vals.mean():.4f}')
print(f'  Std  : {cos_vals.std():.4f}')
print(f'  % > 0.5  : {(cos_vals > 0.5).mean()*100:.2f}%')
print(f'  % > 0.8  : {(cos_vals > 0.8).mean()*100:.2f}%')
print(f'  % > 0.95 : {(cos_vals > 0.95).mean()*100:.2f}%')


In [ ]:
# Step 4: save hetero_data15.pt 

OUTPUT_PATH = 'YOUR_GRAPH_OUTPUT_FILE.pt'

print('\n' + '═'*55)
print('HeteroData v15 construction complete')
print('═'*55)
print(data_v15)
print()
print('Node feature dimensions (after v15 cleaning + normalization):')
print(f'  youtuber.x : {data_v15["youtuber"].x.shape}')
print(f'  video.x    : {data_v15["video"].x.shape}')
print(f'  brand.x    : {data_v15["brand"].x.shape}')
print()
print('Edge counts:')
for edge_type in data_v15.edge_types:
    e = data_v15[edge_type].edge_index
    print(f'  {str(edge_type):<55}: {e.shape[1]}')

torch.save(data_v15, OUTPUT_PATH)
print(f'\nSaved to {OUTPUT_PATH}')
